# 02 - Limpeza e transformacao

Este notebook le a camada bronze, trata tipos e valores ausentes e cria indicadores para a analise.

In [ ]:
from pyspark.sql import functions as F

bronze_table = "bronze_acidentes"
silver_table = "silver_acidentes"

bronze_df = spark.table(bronze_table)
print(f"Linhas lidas da camada bronze: {bronze_df.count():,}")

In [ ]:
clean_df = (
    bronze_df
    .withColumn(
        "data_acidente",
        F.to_date(F.to_timestamp("crash_date", "yyyy-MM-dd'T'HH:mm:ss.SSS")),
    )
    .withColumn("hora_acidente", F.to_timestamp(F.trim("crash_time"), "H:mm"))
    .withColumn("regiao", F.coalesce(F.col("borough"), F.lit("NAO INFORMADA")))
    .withColumn("feridos", F.coalesce(F.col("number_of_persons_injured").cast("double"), F.lit(0.0)))
    .withColumn("mortos", F.coalesce(F.col("number_of_persons_killed").cast("double"), F.lit(0.0)))
    .withColumn("gravidade", F.col("feridos") + F.lit(10.0) * F.col("mortos"))
    .withColumn("dia_semana", F.date_format("data_acidente", "EEEE"))
    .withColumn("hora", F.hour("hora_acidente"))
    .withColumn("ano", F.year("data_acidente"))
)

In [ ]:
selected_columns = [
    "data_acidente",
    "hora_acidente",
    "regiao",
    "feridos",
    "mortos",
    "gravidade",
    "dia_semana",
    "hora",
    "ano",
    "contributing_factor_vehicle_1",
    "contributing_factor_vehicle_2",
    "contributing_factor_vehicle_3",
    "contributing_factor_vehicle_4",
    "contributing_factor_vehicle_5",
]
silver_df = clean_df.select(*selected_columns)
display(silver_df.limit(10))

In [ ]:
silver_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(silver_table)

print(f"Linhas na camada silver: {silver_df.count():,}")
print(f"Dados tratados salvos na tabela: {silver_table}")